# mithril2pq tables

Every table is a DuckDB view over its Parquet files, so nothing is read until a query runs.

- `tables.<name>` / `tables["<name>"]`: the whole table as a pandas DataFrame, read on first access and cached.
- `load(name, columns=..., where=..., limit=...)`: part of a table.
- `sql("select ...")`: any query over the views, as a DataFrame.

With `hex=True` (the default) hash, key and signature columns come back as lowercase hex strings. In SQL,
compare them as blobs (`where hash = from_hex('...')`) or convert with the `hx()` macro.
Large tables (`plutus_data_node`, `tx_output_asset`, `tx_input`, ...) can need tens of GB as a full DataFrame:
select columns, filter, or aggregate in SQL instead.

In [1]:
import os
from collections.abc import Mapping
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

DATA = Path(os.environ.get("MITHRIL2PQ_DATA", "/data"))

con = duckdb.connect()
con.sql(f"SET memory_limit = '{os.environ.get('DUCKDB_MEMORY_LIMIT', '16GB')}'")
con.sql("CREATE OR REPLACE MACRO hx(b) AS lower(hex(b))")

TABLES = sorted(p.name for p in DATA.iterdir() if p.is_dir() and any(p.glob("part-*.parquet")))
# Columns stored as fixed-size binary: hashes, keys, signatures, IP addresses.
HASH_COLUMNS = {}
for _name in TABLES:
    con.sql(f"CREATE OR REPLACE VIEW {_name} AS SELECT * FROM read_parquet('{DATA / _name}/part-*.parquet')")
    _schema = pq.read_schema(next((DATA / _name).glob("part-*.parquet")))
    HASH_COLUMNS[_name] = {f.name for f in _schema if pa.types.is_fixed_size_binary(f.type)}


def _df(rel: duckdb.DuckDBPyRelation) -> pd.DataFrame:
    # Arrow-backed columns keep nullable uint64 values exact instead of turning them into float64.
    return rel.to_arrow_table().to_pandas(types_mapper=pd.ArrowDtype)


def sql(query: str, *params) -> pd.DataFrame:
    """Run a query over the table views and return a DataFrame."""
    return _df(con.sql(query, params=list(params) or None))


def load(name: str, columns=None, where: str | None = None, limit: int | None = None, hex: bool = True) -> pd.DataFrame:
    """Read a table, or part of it.

    columns: list of column names; where: a SQL condition, e.g. "lovelace > 10e12";
    hex: return hash columns as hex strings rather than bytes.
    """
    if name not in HASH_COLUMNS:
        raise KeyError(name)
    cols = columns or [f.name for f in pq.read_schema(next((DATA / name).glob("part-*.parquet")))]
    select = ", ".join(f"hx({c}) AS {c}" if hex and c in HASH_COLUMNS[name] else c for c in cols)
    query = f"SELECT {select} FROM {name}"
    if where:
        query += f" WHERE {where}"
    if limit is not None:
        query += f" LIMIT {int(limit)}"
    return sql(query)


class Tables(Mapping):
    """Every table as a DataFrame, read on first access and cached."""

    def __init__(self, hex: bool = True):
        self.hex = hex
        self._cache: dict[str, pd.DataFrame] = {}

    def __getitem__(self, name: str) -> pd.DataFrame:
        if name not in self._cache:
            self._cache[name] = load(name, hex=self.hex)
        return self._cache[name]

    def __getattr__(self, name: str) -> pd.DataFrame:
        try:
            return self[name]
        except KeyError:
            raise AttributeError(name) from None

    def __dir__(self):
        return [*super().__dir__(), *TABLES]

    def __iter__(self):
        return iter(TABLES)

    def __len__(self):
        return len(TABLES)

    def drop(self, name: str) -> None:
        """Free a cached table."""
        self._cache.pop(name, None)

    def rows(self) -> pd.Series:
        """Row counts from Parquet metadata, without reading any data."""
        return pd.Series(
            {n: con.sql(f"SELECT sum(num_rows) FROM parquet_file_metadata('{DATA / n}/part-*.parquet')").fetchall()[0][0] for n in TABLES},
            name="rows",
        )

    def __repr__(self):
        return f"Tables({len(TABLES)} tables at {DATA}; loaded: {', '.join(self._cache) or 'none'})"


tables = Tables(hex=True)
tables

Tables(32 tables at /data; loaded: none)

In [2]:
tables.rows().sort_values(ascending=False)

plutus_data_node             214117139
tx_output_asset               45124456
tx_input                      35896527
tx_output                     29287652
metadata_node                 21191189
redeemer                      11344938
vkey_witness                   8706299
native_script_node             7514399
tx                             7067473
block_header_vrf               4867862
block_header                   4854850
block                          4854850
tx_script                      2222513
tx_mint                        1419668
tx_required_signer             1380880
plutus_datum_witness           1017427
tx_withdrawal                   563985
certificate                     201848
certificate_credential          198795
script                           33710
param_update_cost_model          14421
vote                              5191
pool_owner                        3221
bootstrap_witness                 3144
pool_registration                 2460
pool_relay               

In [3]:
tables.tx.head()

,hash,block_hash,block_index,valid,fee,ttl,validity_interval_start,total_collateral,auxiliary_data_hash,network_id,script_data_hash,current_treasury_value,donation
0,a8fa4293645facb2a0332f4dfc442dff3fc9ca021c95ee...,e19690d1a8ab6cab8ba342970bac8bc530a21425e011df...,0,True,200000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,e3ca57e8f323265742a8f4e79ff9af884c9ff8719bd4f7...,f4217a065ec70a058315bdbf37f2f8809e3f91deb5edd9...,0,True,237793,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,214bdcf674d45db7ecece7e6c7aa020a3741c0fdcf1379...,6df990843be0e16eec6446ce2cfdf2fb33fe9034c09bb9...,0,True,206597,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,1839dd7385c38e00f19c3937389685bd257ef0a05e1f5c...,1058d71acd8f78910295b2fc4fa22db197c5516e96e3bb...,0,True,205981,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,edb991bbbfcb05dbccaea6660db60bbfe25c52dec08627...,c8f692fae7f77edd10cdd74aa8fb2fa67ac8bb320e9a72...,0,True,304233,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [4]:
# Part of a large table.
load("tx_output", columns=["tx_hash", "output_index", "address", "lovelace"], where="lovelace > 10e12").head()

,tx_hash,output_index,address,lovelace
0,a8fa4293645facb2a0332f4dfc442dff3fc9ca021c95ee...,0,addr_test1vp8cprhse9pnnv7f4l3n6pj0afq2hjm6f7r2...,29999999999800000
1,e3ca57e8f323265742a8f4e79ff9af884c9ff8719bd4f7...,0,addr_test1qp8cprhse9pnnv7f4l3n6pj0afq2hjm6f7r2...,100000000000000
2,e3ca57e8f323265742a8f4e79ff9af884c9ff8719bd4f7...,1,addr_test1qp8cprhse9pnnv7f4l3n6pj0afq2hjm6f7r2...,100000000000000
3,e3ca57e8f323265742a8f4e79ff9af884c9ff8719bd4f7...,2,addr_test1qp8cprhse9pnnv7f4l3n6pj0afq2hjm6f7r2...,100000000000000
4,e3ca57e8f323265742a8f4e79ff9af884c9ff8719bd4f7...,3,addr_test1vp8cprhse9pnnv7f4l3n6pj0afq2hjm6f7r2...,29699998493562207


In [5]:
# Aggregate in SQL instead of loading: blocks per software tag in the header's protocol minor version
# (69 in the low byte is Dingo; see the mithril2pq README for the encoding).
sql("""
    SELECT protocol_minor & 255 AS impl_id, count(*) AS blocks, count(DISTINCT issuer_vkey) AS pools
    FROM block_header
    WHERE protocol_minor IS NOT NULL
    GROUP BY ALL ORDER BY blocks DESC
""")

,impl_id,blocks,pools
0,0,2969288,319
1,2,576820,220
2,3,442369,133
3,1,438224,171
4,6,272879,162
5,7,84489,49
6,8,43412,31
7,69,21318,8
8,67,3150,21
9,65,2901,4


## Plots

Each plot aggregates in DuckDB and loads only the result, so they work on mainnet-sized data.
`epoch(slot)` is a SQL macro for the dataset's network (Byron and Shelley epoch lengths from each network's genesis).

In [ ]:
import json

import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

NETWORK = json.loads((DATA / "_mithril2pq.json").read_text())["network"]
# (Byron epochs, Byron slots per epoch, Shelley slots per epoch)
EPOCHS = {"mainnet": (208, 21600, 432000), "preprod": (4, 21600, 432000), "preview": (0, 21600, 86400)}
_byron_epochs, _byron_len, _shelley_len = EPOCHS[NETWORK]
_byron_slots = _byron_epochs * _byron_len
con.sql(f"""
    CREATE OR REPLACE MACRO epoch(s) AS CASE WHEN s < {_byron_slots} THEN s // {_byron_len}
        ELSE {_byron_epochs} + (s - {_byron_slots}) // {_shelley_len} END
""")
sns.set_theme(style="whitegrid")
NETWORK

### Blocks and transactions per epoch (matplotlib)

In [ ]:
per_epoch = sql("""
    SELECT coalesce(epoch(slot), ebb_epoch) AS epoch, count(*) AS blocks, sum(tx_count) AS txs
    FROM block GROUP BY ALL ORDER BY epoch
""").astype(float)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(per_epoch.epoch, per_epoch.blocks, color="tab:blue", label="blocks")
ax.set_xlabel("epoch")
ax.set_ylabel("blocks", color="tab:blue")
ax2 = ax.twinx()
ax2.plot(per_epoch.epoch, per_epoch.txs, color="tab:orange", label="transactions")
ax2.set_ylabel("transactions", color="tab:orange")
ax2.grid(False)
ax.set_title(f"{NETWORK}: blocks and transactions per epoch")
fig.tight_layout()

### Fee distribution (seaborn)

Fees are binned in SQL on a log scale (20 bins per decade); the histogram is weighted by the bin counts.
Both axes are logarithmic so the tail of large fees stays visible.

In [ ]:
fees = sql("""
    SELECT power(10, floor(log10(fee) * 20) / 20) AS fee, count(*) AS txs
    FROM tx WHERE fee > 0 GROUP BY ALL
""").astype(float)

fig, ax = plt.subplots(figsize=(12, 4))
sns.histplot(fees, x="fee", weights="txs", log_scale=(True, True), bins=80, element="step", fill=False, ax=ax)
ax.set_xlabel("fee (lovelace)")
ax.set_ylabel("transactions")
ax.set_title(f"{NETWORK}: transaction fees")
fig.tight_layout()

### Dingo vs Haskell node blocks per epoch (plotly)

Dingo sets the low byte of the header's protocol minor version to 69 and leaves bits 31-30 clear.
Blocks without a protocol version in the header (Byron) are left out.

In [ ]:
impl = sql("""
    SELECT epoch(b.slot) AS epoch,
           CASE WHEN h.protocol_minor & 255 = 69 AND h.protocol_minor >> 30 = 0 THEN 'Dingo' ELSE 'Haskell node' END AS node,
           count(*) AS blocks
    FROM block_header h JOIN block b ON b.hash = h.block_hash
    WHERE h.protocol_minor IS NOT NULL
    GROUP BY ALL ORDER BY epoch, node
""")
# One row per epoch and node, zero-filled: stacked areas need every series at every x.
impl = impl.astype({"epoch": int, "blocks": int}).pivot(index="epoch", columns="node", values="blocks").fillna(0)
impl = impl.stack().rename("blocks").reset_index()
impl["share"] = impl.blocks / impl.groupby("epoch").blocks.transform("sum")

px.area(
    impl,
    x="epoch", y="share", color="node", hover_data=["blocks"],
    title=f"{NETWORK}: share of blocks by node implementation",
).update_yaxes(tickformat=".0%")